# LLM-for-Metadata-Harvesting

This notebook demonstrates the use of Large Language Models (LLMs) for automated metadata extraction from web-based dataset portals.  
It showcases an experiment on the [Actual probability distribution for Quercus robur (2000–2020)](https://stac.ecodatacube.eu/veg_quercus.robur_anv.eml/collection.json?.language=en) dataset, using a combination of web scraping and LLM-based entity extraction to populate metadata fields according to the Croissant data standard.

Key features:
- Web scraping utilities for extracting full page text from dataset portals
- Environment configuration for flexible API and model usage
- LLM client support for OpenAI and Gemini models (with extensibility for custom clients)
- Automated extraction of core metadata fields such as description, license, creator, keywords, and more

The results of the experiment are presented at the end of the notebook.  
Some code cells are included for future development and may not be directly relevant to this specific experiment.


In [ ]:
from dotenv import load_dotenv
from llm_metadata_harvester.harvester_operations import metadata_harvest
from llm_metadata_harvester.standards import LTER_LIFE_STANDARD

load_dotenv("../.env")

extracted_metadata = await metadata_harvest(
    model_name="surf-openai/gpt-oss-120b",
    url="https://dataverse.nioz.nl/dataset.xhtml?persistentId=doi:10.25850/nioz/7b.b.ug",
    metadata_standard=LTER_LIFE_STANDARD,
)

extracted_metadata

In [1]:
from dotenv import load_dotenv
from llm_metadata_harvester.structured_harvester_operations import metadata_harvest_structured
from llm_metadata_harvester.standards import LTER_LIFE_STANDARD

load_dotenv("../.env")

extracted_metadata_structured = await metadata_harvest_structured(
    model_name="surf-Qwen/Qwen3.6-35B-A3B-FP8",
    url="https://dataverse.nioz.nl/dataset.xhtml?persistentId=doi:10.25850/nioz/7b.b.ug",
    metadata_standard=LTER_LIFE_STANDARD,
)

extracted_metadata_structured

Extracting full page text...
Extracting metadata (structured output)...


{'Metadata date': '2023-12-06',
 'Metadata language': 'eng',
 'Responsible organization metadata': 'NIOZ',
 'Landing page': 'https://doi.org/10.25850/nioz/7b.b.ug',
 'Title': 'SIBES dataset',
 'Description': 'Here, we present SIBES-data on median grain size (micrometer) and mud content (%) of sediment, and abundances (#) and biomass (g) of benthic invertebrate found at sampling stations on intertidal mudflats of the Dutch Wadden Sea. SIBES stands for Synoptic Intertidal BEnthic Survey and is a long-term ecological time-series performed by NIOZ. It is aimed at understanding ecological processes and sediment dynamics of intertidal mudflats related to benthic invertebrates. Details of the sampling method, laboratory protocols and data curation can be found in the accompanying paper in the journal "Scientific Data" (https://doi.org/10.1038/s41597-025-04540-9). In summary, roughly 4,500 stations are sampled yearly from 2008 and onwards. The most recent 3 years are under embargo, but availab

In [2]:
from dotenv import load_dotenv
from llm_metadata_harvester.harvester_operations import metadata_harvest
from llm_metadata_harvester.standards import LTER_LIFE_STANDARD

load_dotenv("../.env")

extracted_metadata = await metadata_harvest(
    model_name="surf-Qwen/Qwen3.6-35B-A3B-FP8",
    url="https://dataverse.nioz.nl/dataset.xhtml?persistentId=doi:10.25850/nioz/7b.b.ug",
    metadata_standard=LTER_LIFE_STANDARD,
    fields=["Title", "Description", "License", "Keywords"],
)

extracted_metadata

Extracting full page text...
Extracting entities from text...
Converting extracted nodes to metadata...


{'Title': 'SIBES dataset. A long-term ecological time-series dataset from NIOZ containing data on benthic invertebrates and sediment characteristics from the Dutch Wadden Sea."|;',
 'Description': 'SIBES dataset. This dataset presents median grain size, mud content, and abundance/biomass of benthic invertebrates sampled from intertidal mudflats in the Dutch Wadden Sea."|;',
 'License': 'SIBES dataset. The dataset is associated with NIOZ and requires citation of the accompanying paper in Scientific Data for use."|;',
 'Keywords': 'benthic invertebrates, biodiversity, biomass, intertidal mudflats, macrozoobenthos, median grain size, mud content, sediment, Wadden Sea, SIBES"|;'}

# Scrap from the web portal of the dataset

You can use own-defined function to get the data from website, or use the pre-defined function in webutils.

Mind that you should check the `robots.txt` first to make sure if it is legal or allowed to scrap from this website.

In [ ]:
from llm_metadata_harvester.webutils import extract_full_page_text
import nest_asyncio

url = "https://stac.ecodatacube.eu/veg_quercus.robur_anv.eml/collection.json?.language=en"

# Apply nest_asyncio to allow asyncio.run() in Jupyter
nest_asyncio.apply()

# Run the async function
full_text = await extract_full_page_text(url)

# Optionally display or save it
print(full_text[:10])  # Print the first 100 characters

Alternatively, you can download the HTML of the webpage and read it as a string.

In [ ]:
file_path = "./webpages/Actual probability distribution for Quercus robur (2000–2020).html"

with open(file_path, "r") as file:
    full_text = file.read()

## 🔧 Environment Configuration

To configure your API keys or other environment variables, you can use a `.env` file or set them directly in your shell.

### 📄 Using a `.env` File

Place the `.env` file in **one** of the following locations:

- The **root directory** of your project  
- The **same directory** as the script you're running  
- Or any directory, **as long as it's the current working directory**

> ℹ️ The `load_dotenv()` function automatically looks for a `.env` file in the current working directory by default. The harvest cells above load `../.env` from the project root when the notebook is run from `examples/`.

#### 💡 Example `.env` File
```env
OPENAI_API_KEY=your_openai_api_key_here
GEMINI_API_KEY=your_gemini_api_key_here
SURF_API_KEY=your_surf_api_key_here
```

### Using global environment variable

Alternatively, you can set environment variables directly in your shell:

```bash
export OPENAI_API_KEY=your_openai_api_key_here
export GEMINI_API_KEY=your_gemini_api_key_here
export SURF_API_KEY=your_surf_api_key_here
```

In [ ]:
from tqdm import tqdm
from llm_metadata_harvester.harvester_operations import extract_entities
from llm_metadata_harvester.llm_client import LLMClient
from dotenv import load_dotenv

# can put your .env file in the root of the project
# or in the same directory as this script
# or set the environment variables directly in your shell
load_dotenv("../.env")

## Metadata Fields

The metadata fields defined below follow the **Croissant data standard**.

In [ ]:
# Define the metadata fields and their descriptions
# These fields are from croissant data standard
meta_field_dict = {
    "description": "Description of the dataset.",
    "license": "The license of the dataset. Croissant recommends using the URL of a known license, e.g., one of the licenses listed at https://spdx.org/licenses/.",
    "name": "The name of the dataset.",
    "creator": "The creator(s) of the dataset.",
    "datePublished": "The date the dataset was published.",
    "keywords": "A set of keywords associated with the dataset, either as free text, or a DefinedTerm with a formal definition.",
    "publisher": "The publisher of the dataset, which may be distinct from its creator.",
    "sameAs": "The URL of another Web resource that represents the same dataset as this one.",
    "dateModified": "The date the dataset was last modified.",
    "inLanguage": "The language(s) of the content of the dataset."
}

If you want to use the lter-life standard, you can do

In [ ]:
from llm_metadata_harvester.standards import LTER_LIFE_STANDARD

## LLM Client Support

The `llm client` currently supports **OpenAI** and **Gemini** models.

To use other models, you can define your own LLM client class.  
Your custom class should implement a `chat` method that returns a string as the LLM response.

In [ ]:
from llm_metadata_harvester.llm_client import LLMClient
llm = LLMClient(model_name="gemini-2.5-flash", temperature=0.0)

In [ ]:
from llm_metadata_harvester.llm_client import LLMClient

llm = LLMClient(model_name="surf-Qwen 2.5 Coder 32B Instruct AWQ", temperature=0.0)

In [ ]:
messages=[
            {
                "role": "system",
                "content": "You are an AI."
            },
            {
                "role": "user",
                "content": "halo"
            }
]
llm.chat(messages)

The output of `extract_entities` looks like a list of lists of dictionaries, where each dictionary is structured like this:

```python
'license': [{'entity_name': 'license',
             'entity_value': 'CC-BY-SA-4.0',
             'source_id': 'chunk_0',
             'file_path': 'unknown_source'}]
```